In [11]:
from pathlib import Path
import cv2
import numpy as np

root = Path("/work/tesi_cmininno/dataset/CholecSeg8k")

image_path = next(root.glob("video*/video*/*_endo.png"))
mask_path = Path(
    str(image_path).replace(
        "_endo.png",
        "_endo_watershed_mask.png"
    )
)

print("Image:", image_path)
print("Mask :", mask_path)

image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)

print("\nIMAGE")
print("shape:", image.shape)
print("dtype:", image.dtype)

print("\nMASK")
print("shape:", mask.shape)
print("dtype:", mask.dtype)
print("unique:", np.unique(mask))

Image: /work/tesi_cmininno/dataset/CholecSeg8k/video55/video55_00508/frame_586_endo.png
Mask : /work/tesi_cmininno/dataset/CholecSeg8k/video55/video55_00508/frame_586_endo_watershed_mask.png

IMAGE
shape: (480, 854, 3)
dtype: uint8

MASK
shape: (480, 854, 3)
dtype: uint8
unique: [ 11  12  13  21  22  31  32  50 255]


In [12]:
if mask.ndim == 3:
    print("Mask is RGB/BGR")
    for c in range(mask.shape[2]):
        print(f"Channel {c}:", np.unique(mask[:, :, c])[:50])
# else:
    print("Mask is grayscale")

Mask is RGB/BGR
Channel 0: [ 11  12  13  21  22  31  32  50 255]
Channel 1: [ 11  12  13  21  22  31  32  50 255]
Channel 2: [ 11  12  13  21  22  31  32  50 255]
Mask is grayscale


In [13]:
print("Image size:", image.shape[:2])
print("Mask size :", mask.shape[:2])

Image size: (480, 854)
Mask size : (480, 854)


In [14]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("../")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.cholecseg8k import CholecSeg8kDataset

In [15]:
DATASET_ROOT = "/work/tesi_cmininno/dataset/CholecSeg8k"

print(Path(DATASET_ROOT).exists())
print(Path(DATASET_ROOT))

True
/work/tesi_cmininno/dataset/CholecSeg8k


In [16]:
root = Path(DATASET_ROOT)

video_dirs = sorted(
    p for p in root.iterdir()
    if p.is_dir() and p.name.startswith("video")
)

print("Number of videos:", len(video_dirs))

for video_dir in video_dirs:
    print(video_dir.name)

Number of videos: 17
video01
video09
video12
video17
video18
video20
video24
video25
video26
video27
video28
video35
video37
video43
video48
video52
video55


In [17]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

SEED = 42

In [18]:
dataset = CholecSeg8kDataset(
    root=DATASET_ROOT
)

print("Total samples:", len(dataset))
print("Videos:", dataset.video_ids)

KeyboardInterrupt: 

In [ ]:
train, val, test = dataset.create_split(
    strategy="percentage",
    train_ratio=0.70,
    val_ratio=0.30,
    test_ratio=0,
    seed=42,
    stratified=True,
)


DATASET SPLIT

Train:
  Groups : [1, 9, 12, 20, 26, 27, 28, 35, 37, 52, 55]
  Frames : 5360

Validation:
  Groups : [17, 18, 24, 25, 43, 48]
  Frames : 2720

Total:
  Groups : 17
  Frames : 8080


In [ ]:
import torch

from src.models.mask2former_wrapper import Mask2FormerWrapper


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# Configuration
# ============================================================

NUM_CLASSES = 13
CHECKPOINT = "facebook/mask2former-swin-base-ade-semantic"


# ============================================================
# Model
# ============================================================

model = Mask2FormerWrapper(
    checkpoint=CHECKPOINT,
    num_labels=NUM_CLASSES,
)

model = model.to(device)

print(model)

/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/homes/cmininno/tesi/tesi_venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of Mask2FormerForUniversalSegmentation were not initialized from the model checkpoint at facebook/mask2former-swin-base-ade-semantic and are newly initialized because the shapes did not match:
- class_predictor.bias: found shape torch.Size([151]) in the checkpoint and torch.Size([14]) in the model instantiated
- class_predictor.weight: found shape torch.Size([151, 256]) in the checkpoint and torch.Size([14, 256]) in

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import torch

from monai.losses import DiceCELoss
from monai.metrics import (
    DiceMetric,
    MeanIoU,
    HausdorffDistanceMetric,
)


# ============================================================
# Configuration
# ============================================================

MAX_EPOCHS = 50
LR = 1.0e-4
WEIGHT_DECAY = 1.0e-5


# ============================================================
# Loss
# ============================================================

loss_fn = DiceCELoss(
    include_background=True,
    to_onehot_y=True,
    softmax=True,
    lambda_dice=0.5,
    lambda_ce=0.5,
)


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)


# ============================================================
# Mixed precision
# ============================================================

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)


# ============================================================
# Metrics
# ============================================================

metrics = {
    "dice": DiceMetric(
        include_background=True,
        reduction="mean",
    ),
    "iou": MeanIoU(
        include_background=True,
        reduction="mean",
    ),
    "hd95": HausdorffDistanceMetric(
        include_background=True,
        percentile=95,
        reduction="mean",
    ),
}

print("Loss:", loss_fn)
print("Optimizer:", optimizer)
print("Scheduler:", scheduler)
print("Scaler:", scaler)
print("Metrics:", )

Loss: DiceCELoss(
  (dice): DiceLoss()
  (cross_entropy): CrossEntropyLoss()
  (binary_cross_entropy): BCEWithLogitsLoss()
)
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 1e-05
)
Scheduler: <torch.optim.lr_scheduler.CosineAnnealingLR object at 0x7fa52d1d7550>
Scaler: <torch.cuda.amp.grad_scaler.GradScaler object at 0x7fa514267e10>
Metrics:


In [ ]:
from src.metrics.metric_manager import MetricManager
from src.utils.wandb_logging import WandbLogger


In [ ]:
if hasattr(dataset, "CLASS_NAMES"):
    class_names = {i: name for i, name in enumerate(dataset.CLASS_NAMES)}
else:
    raise ValueError(f"CLASS_NAMES attribute is missing in dataset")

In [ ]:
metric_manager = MetricManager(
    metrics=metrics,
    num_classes=13,
    dataset_name="cholecseg8k",
    class_names=class_names,
)

In [ ]:
logger = WandbLogger(
    enabled=True,
    project="surgical_scene_segmentation",
    name="cholecseg8k_baseline",
    config={
        "dataset": "cholecseg8k",
        "model": "Mask2Former",
        "num_classes": 13,
        "batch_size": 4,
        "accumulation_steps": 3,
        "max_epochs": 50,
        "lr": 1e-4,
        "weight_decay": 1e-5,
    },
    tags=["cholecseg8k", "baseline"],
    mode="online",
)

wandb: Currently logged in as: mininnoclaudio (mininnoclaudio-unimore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
from src.engine.trainer import Trainer
from hydra import initialize, compose
from omegaconf import OmegaConf

config_path = "/homes/cmininno/tesi/surgical_scene_segmentation/configs/training/baseline/cholec_seg8k.yaml"

cfg = OmegaConf.load(config_path)

print(OmegaConf.to_yaml(cfg))

train:
  batch_size: 4
  accumulation_steps: 3
  max_epochs: 50
  eval_freq: 1
  num_workers: 4
  pin_memory: true
  precision: 16-mixed
  initialization: scratch
  early_stopping:
    enabled: true
    patience: 10
    monitor: loss
    mode: min
  optimizer:
    _target_: torch.optim.AdamW
    lr: 0.0001
    weight_decay: 1.0e-05
  scheduler:
    _target_: torch.optim.lr_scheduler.CosineAnnealingLR
    T_max: ${train.max_epochs}
  scaler:
    _target_: torch.cuda.amp.GradScaler
  loss:
    _target_: monai.losses.DiceCELoss
    include_background: true
    to_onehot_y: true
    softmax: true
    lambda_dice: 0.5
    lambda_ce: 0.5
evaluation:
  metrics:
    dice:
      _target_: monai.metrics.DiceMetric
      include_background: true
      reduction: mean
    iou:
      _target_: monai.metrics.MeanIoU
      include_background: true
      reduction: mean
    hd95:
      _target_: monai.metrics.HausdorffDistanceMetric
      include_background: true
      percentile: 95
      reduction: 

In [ ]:
from src.datasets.transformation.transform_dataset import TransformDataset
from src.datasets.transformation.factory import TransformFactory



In [ ]:
config_path_data = "/homes/cmininno/tesi/surgical_scene_segmentation/configs/dataset/baseline/cholec_seg8k.yaml"

cfg_data = OmegaConf.load(config_path_data)

print(OmegaConf.to_yaml(cfg_data))

name: cholecseg8k
data:
  name: cholecseg8k
  dataset:
    _target_: src.datasets.cholecseg8k.CholecSeg8kDataset
    root: ${paths.data}/CholecSeg8k
  num_classes: 13
  split:
    strategy: percentage
    group_by: video
    seed: 42
    stratified: true
    train_ratio: 0.7
    val_ratio: 0.3
    test_ratio: 0.0
preprocessing:
  image:
    size:
    - 240
    - 427
    channel_first: true
    dtype: float32
    normalize: true
  mask:
    channel_first: true
    dtype: long
  padding:
    enabled: true
    divisor: 16
augmentation:
  train:
    enabled: true
    horizontal_flip:
      enabled: true
      probability: 0.5
    scaling:
      enabled: true
      range:
      - -0.1
      - 0.1
      probability: 1.0
    gaussian_blur:
      enabled: true
      probability: 1.0
      kernel_sizes:
      - 3
      - 5
      - 7
      sigmas:
      - 0.8
      - 1.1
      - 1.4
    random_crop:
      enabled: true
      size:
      - 256
      - 256
  validation:
    enabled: false
  test:


In [ ]:
transform_factory = TransformFactory(cfg_data)

train_transform = transform_factory.build_train_transform()
val_transform = transform_factory.build_val_transform()
test_transform = transform_factory.build_test_transform() if test is not None else None


In [ ]:
print(train_transform.transforms)

(<monai.transforms.utility.dictionary.EnsureChannelFirstd object at 0x7fa504d9de50>, <monai.transforms.utility.dictionary.EnsureChannelFirstd object at 0x7fa504d9de90>, <monai.transforms.spatial.dictionary.Resized object at 0x7fa504e88150>, <monai.transforms.croppad.dictionary.SpatialPadd object at 0x7fa507195e10>, <monai.transforms.croppad.dictionary.SpatialPadd object at 0x7fa5071f7890>, <monai.transforms.intensity.dictionary.ScaleIntensityd object at 0x7fa5071a5990>, <monai.transforms.utility.dictionary.EnsureTyped object at 0x7fa5071a7e50>, <monai.transforms.spatial.dictionary.RandFlipd object at 0x7fa504e8b8d0>, <monai.transforms.spatial.dictionary.RandAffined object at 0x7fa504e905d0>, <monai.transforms.intensity.dictionary.RandGaussianSmoothd object at 0x7fa5071d0950>, <monai.transforms.croppad.dictionary.SpatialPadd object at 0x7fa504d83150>, <monai.transforms.croppad.dictionary.SpatialPadd object at 0x7fa504e91850>, <monai.transforms.croppad.dictionary.RandSpatialCropd object 

In [ ]:
train_dataset = TransformDataset(train, transform=train_transform)
val_dataset = TransformDataset(val, transform=val_transform)


In [ ]:
from src.utils.repro import seed_worker

In [ ]:
generator = torch.Generator()
generator.manual_seed(42)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train.batch_size,
    shuffle=True,
    num_workers=cfg.train.num_workers,
    pin_memory=cfg.train.pin_memory,
    worker_init_fn=seed_worker,
    generator=generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.train.batch_size,
    shuffle=False,
    num_workers=cfg.train.num_workers,
    pin_memory=cfg.train.pin_memory,
    worker_init_fn=seed_worker,
    generator=generator,
)

In [ ]:
batch = next(iter(train_loader))
print(batch.keys())

dict_keys(['image', 'mask', 'video', 'video_id', 'clip', 'frame'])


In [ ]:
image= batch["image"]
mask = batch["mask"]

print("images:", image.shape)
print("masks:", mask.shape)


images: torch.Size([4, 3, 256, 256])
masks: torch.Size([4, 1, 256, 256])


In [ ]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=loss_fn,
    scaler=scaler,
    metric_manager=metric_manager,
    cfg=cfg,
    device=device,
    logger=logger,
)


In [ ]:
batch = next(iter(train_loader))

images = batch["image"].to(device)
masks = batch["mask"].to(device)

print("images:", images.shape)
print("masks:", masks.shape)

mask_labels, class_labels = model._build_targets(masks)

print("mask_labels:")
for i, x in enumerate(mask_labels):
    print(i, x.shape)

print("class_labels:")
for i, x in enumerate(class_labels):
    print(i, x.shape)

images: torch.Size([4, 3, 256, 256])
masks: torch.Size([4, 1, 256, 256])
mask_labels:
0 torch.Size([6, 256, 256])
1 torch.Size([5, 256, 256])
2 torch.Size([5, 256, 256])
3 torch.Size([8, 256, 256])
class_labels:
0 torch.Size([6])
1 torch.Size([5])
2 torch.Size([5])
3 torch.Size([8])


In [ ]:

train_results = trainer.fit()


KeyboardInterrupt: 

In [ ]:
image = sample["image"]
mask = sample["mask"]

print("Image:")
print("  shape:", image.shape)
print("  dtype:", image.dtype)

print("\nMask:")
print("  shape:", mask.shape)
print("  dtype:", mask.dtype)

print("\nMetadata:")
print("  video:", sample["video"])
print("  clip :", sample["clip"])
print("  frame:", sample["frame"])

In [ ]:
unique_classes = np.unique(mask)

print("Classes present:", unique_classes)
print("Number of classes:", len(unique_classes))

In [ ]:
print("Original -> semantic class")

for original_value, class_id in sorted(
    dataset.LABEL_MAP.items()
):
    print(
        f"{original_value:>3} -> "
        f"{class_id:>2} "
        f"({dataset.CLASS_NAMES[class_id]})"
    )

In [ ]:
from src.metrics.metric_manager import MetricManager
import monai.metrics

In [ ]:
metrics_dict = {
    "dice": monai.metrics.DiceMetric(
        include_background=True, 
        reduction="mean"
    ),
    "iou": monai.metrics.MeanIoU(
        include_background=True, 
        reduction="mean"
    ),
    "hd95": monai.metrics.HausdorffDistanceMetric(
        include_background=True, 
        percentile=95, 
        reduction="mean"
    )
}

In [ ]:
class_names = {
    0: "background",
    1: "abdominal_wall",
    2: "liver",
    3: "gastrointestinal_tract",
    4: "fat",
    5: "grasper",
    6: "connective_tissue",
    7: "blood",
    8: "cystic_duct",
    9: "l_hook_electrocautery",
    10: "gallbladder",
    11: "hepatic_vein",
    12: "liver_ligament"
}

In [ ]:


metric_manager = MetricManager(
    metrics=metrics_dict,
    num_classes=13,
    dataset_name="cholecseg8k",
    class_names=class_names
)

In [ ]:
metric_manager.class_names

In [ ]:
plt.figure(figsize=(16, 5))

plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title(
    f"Image - {sample['video']} / {sample['frame']}"
)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="tab20")
plt.title("Semantic mask")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.imshow(image)
plt.imshow(
    np.ma.masked_where(mask == 0, mask),
    cmap="tab20",
    alpha=0.45,
)

plt.title(
    f"Overlay - {sample['video']} / {sample['frame']}"
)

plt.axis("off")
plt.show()

In [ ]:
present_classes = np.unique(mask)

print("Classes present in this frame:")

for class_id in present_classes:
    print(
        f"{class_id:2d} -> "
        f"{dataset.CLASS_NAMES[class_id]}"
    )

In [ ]:
def show_sample(dataset, index):

    sample = dataset[index]

    image = sample["image"]
    mask = sample["mask"]

    present_classes = np.unique(mask)

    print(
        f"Index: {index}\n"
        f"Video: {sample['video']}\n"
        f"Clip : {sample['clip']}\n"
        f"Frame: {sample['frame']}\n"
        f"Image: {image.shape}, {image.dtype}\n"
        f"Mask : {mask.shape}, {mask.dtype}\n"
    )

    print("Classes:")

    for class_id in present_classes:
        print(
            f"  {class_id:2d}: "
            f"{dataset.CLASS_NAMES[class_id]}"
        )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].imshow(image)
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="tab20")
    axes[1].set_title("Mask")
    axes[1].axis("off")

    axes[2].imshow(image)
    axes[2].imshow(
        np.ma.masked_where(mask == 0, mask),
        cmap="tab20",
        alpha=0.45,
    )
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
show_sample(dataset, 50)

In [ ]:
show_sample(dataset, 7000)

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
)

In [ ]:
batch = next(iter(loader))

print(batch.keys())